# Project 04 — Complete Longformer Training and Evaluation Pipeline

This notebook upgrades Project 04 from a functional demo to a portfolio-grade experiment. It:

1. downloads the official QASPER v0.3 dataset;
2. keeps only contiguous extractive answers compatible with span-based QA;
3. optionally fine-tunes Longformer on the prepared QASPER subset;
4. compares **BERT truncated at 512 tokens**, the published **Longformer SQuAD checkpoint**, and the **QASPER-fine-tuned Longformer**;
5. calculates Exact Match, Token F1, evidence recovery, continuous evidence token recall, latency, GPU memory, confidence association, context-length performance, and answer-position performance;
6. writes the real outputs to JSON, CSV, PNG, Markdown, `README.md`, and `MODEL_CARD.md`.

> Run every cell in order. Do not publish performance claims until this notebook finishes successfully on your machine.

## 1. Environment setup

Install CUDA-enabled PyTorch using the selector on the official PyTorch website first. Then run:

```bash
pip install -r requirements.txt -r requirements-evaluation.txt
```

The notebook checks CUDA and recommends a training profile based on available GPU memory.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "README.md").exists():
            return candidate
    # Notebook is normally inside notebooks/
    candidate = start.parent if start.name == "notebooks" else start
    if (candidate / "src").exists():
        return candidate
    raise FileNotFoundError("Open this notebook from inside the Project 04 folder.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Experiment configuration

The **portfolio** profile is the recommended default for a strong RTX workstation. The notebook can be rerun with `smoke`, `full`, or `high-vram` after the first successful run.

In [ ]:
SEED = 42
PROFILE = "portfolio"          # smoke | portfolio | full | high-vram
RUN_FINE_TUNING = True         # Keep True for the 9/10 portfolio version
EVAL_EXAMPLES = 120            # Use at least 100 for publishable portfolio evidence
LONGFORMER_MAX_LENGTH = 2048   # Increase to 3072/4096 only after confirming VRAM headroom
LONGFORMER_STRIDE = 256
FORCE_DATA_DOWNLOAD = False

BASE_LONGFORMER_ID = "valhalla/longformer-base-4096-finetuned-squadv1"
BERT_BASELINE_ID = "deepset/bert-base-cased-squad2"
FINE_TUNED_MODEL_DIR = PROJECT_ROOT / "models" / "qasper-longformer"

print({
    "profile": PROFILE,
    "run_fine_tuning": RUN_FINE_TUNING,
    "evaluation_examples": EVAL_EXAMPLES,
    "longformer_max_length": LONGFORMER_MAX_LENGTH,
    "longformer_stride": LONGFORMER_STRIDE,
})

## 3. GPU and software diagnostics

In [ ]:
import torch
import transformers

from src.qasper_training import PROFILES, recommend_profile

gpu_report = recommend_profile()
gpu_report.update({
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "torch_cuda_version": torch.version.cuda,
})
display(gpu_report)

if not torch.cuda.is_available() and RUN_FINE_TUNING:
    raise RuntimeError(
        "CUDA-enabled PyTorch is not active. Install the correct PyTorch CUDA build before fine-tuning."
    )

print("Selected training profile:")
display(PROFILES[PROFILE].to_dict())

## 4. Download and prepare the QASPER extractive subset

QASPER contains free-form, yes/no, unanswerable, multi-span, and extractive answers. Because this project uses an extractive QA head, the preparation step keeps only examples with at least one contiguous answer span that can be located in the reconstructed paper text. The filtering counts are written to `outputs/qasper_dataset_summary.json`.

In [ ]:
from src.qasper_dataset import prepare_qasper_dataset

profile = PROFILES[PROFILE]
dataset_summary = prepare_qasper_dataset(
    PROJECT_ROOT,
    train_limit=profile.train_examples,
    validation_limit=None,  # retain the full extractive validation pool before sampling
    seed=SEED,
    force_download=FORCE_DATA_DOWNLOAD,
)
display(dataset_summary)

In [ ]:
from src.qasper_dataset import load_prepared_split

processed_dir = PROJECT_ROOT / "data" / "processed" / "qasper"
train_frame = load_prepared_split(processed_dir / "qasper_train_extractive.parquet")
validation_frame = load_prepared_split(processed_dir / "qasper_validation_extractive.parquet")

print("Prepared train shape:", train_frame.shape)
print("Prepared validation shape:", validation_frame.shape)
display(train_frame[["title", "question", "primary_answer", "document_character_count"]].head())

## 5. Fine-tune Longformer on QASPER

This step continues training from the published SQuAD-fine-tuned Longformer checkpoint. The model is saved locally under `models/qasper-longformer/`, while training history and metadata are saved under `outputs/`. Large weights are intentionally excluded from Git and can later be uploaded to a Hugging Face model repository.

In [ ]:
from src.qasper_training import fine_tune_longformer

if RUN_FINE_TUNING:
    training_summary = fine_tune_longformer(
        train_frame=train_frame,
        validation_frame=validation_frame,
        project_root=PROJECT_ROOT,
        profile_name=PROFILE,
        base_model_id=BASE_LONGFORMER_ID,
        seed=SEED,
    )
else:
    training_summary = {"status": "not_run", "reason": "RUN_FINE_TUNING=False"}

display(training_summary)

## 6. Select a reproducible evaluation sample

The sample is balanced across answer-position bands so the benchmark contains answers near the beginning, middle, and end of papers. This is important when comparing a first-512-token BERT baseline against sliding-window Longformer inference.

In [ ]:
from src.qasper_dataset import select_evaluation_sample

sample = select_evaluation_sample(validation_frame, maximum_examples=EVAL_EXAMPLES, seed=SEED)
sample_path = processed_dir / "qasper_evaluation_sample.parquet"
sample.to_parquet(sample_path, index=False)

print("Evaluation sample:", sample.shape)
display(sample["answer_position_band"].value_counts(dropna=False).sort_index())

## 7. Define the benchmark models

- **BERT truncated 512:** standard short-context baseline; only the first model window is used.
- **Longformer SQuAD sliding windows:** published Longformer checkpoint over the full document using overlapping windows.
- **Longformer QASPER fine-tuned:** project-trained checkpoint, included only when fine-tuning completed.

In [ ]:
from src.benchmark_models import BenchmarkSpec

specs = [
    BenchmarkSpec(
        name="BERT truncated 512",
        model_id=BERT_BASELINE_ID,
        strategy="truncate",
        max_length=512,
        stride=0,
        inference_batch_size=4,
        description="Standard BERT QA baseline restricted to the first 512-token input.",
    ),
    BenchmarkSpec(
        name="Longformer SQuAD sliding windows",
        model_id=BASE_LONGFORMER_ID,
        strategy="sliding",
        max_length=LONGFORMER_MAX_LENGTH,
        stride=LONGFORMER_STRIDE,
        inference_batch_size=1,
        description="Published Longformer QA checkpoint evaluated across the full document.",
    ),
]

if (FINE_TUNED_MODEL_DIR / "config.json").exists():
    specs.append(
        BenchmarkSpec(
            name="Longformer QASPER fine-tuned",
            model_id=str(FINE_TUNED_MODEL_DIR),
            strategy="sliding",
            max_length=LONGFORMER_MAX_LENGTH,
            stride=LONGFORMER_STRIDE,
            inference_batch_size=1,
            description="Project-trained Longformer checkpoint fine-tuned on extractive QASPER examples.",
        )
    )

for spec in specs:
    display(spec.to_dict())

## 8. Run real model inference and calculate metrics

This is the time-consuming stage. Every model prediction is produced by the actual Transformer checkpoint. The benchmark records answer spans, supporting paragraphs, confidence proxy, window count, latency, and peak GPU memory. Exact Match and Token F1 use the best score across all valid reference answers.

In [ ]:
from src.advanced_evaluation import score_predictions
from src.benchmark_models import TransformerQABenchmarkRunner, evaluate_runner

scored_by_model = {}
for spec in specs:
    print(f"\n=== {spec.name} ===")
    runner = TransformerQABenchmarkRunner(spec)
    predictions = evaluate_runner(
        runner,
        sample,
        progress_callback=lambda done, total, name: print(
            f"\r{name}: {done}/{total}", end="", flush=True
        ),
    )
    print()
    scored = score_predictions(predictions)
    scored_by_model[spec.name] = scored
    runner.unload()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    display(scored[[
        "question", "predicted_answer", "exact_match", "token_f1",
        "evidence_recovered", "latency_seconds", "answer_position_bucket"
    ]].head())

## 9. Save JSON, CSV, PNG, Markdown, README, and model-card results

This cell is the equivalent of the automated output-generation stage used in Project 08. It writes the benchmark artifacts directly into `outputs/` and updates marked sections in `README.md` and `MODEL_CARD.md`.

In [ ]:
from src.results_reporting import generate_complete_report

manifest = generate_complete_report(
    project_root=PROJECT_ROOT,
    scored_by_model=scored_by_model,
    dataset_summary=dataset_summary,
    training_summary=training_summary,
)
display(manifest)

## 10. Controlled context-length experiment

Natural QASPER papers are usually long, so a natural-only analysis may leave the shorter buckets underrepresented. This controlled experiment creates answer-preserving contexts of approximately **384, 768, 1,536, 3,072, and 4,608 tokens** around the same questions. It measures how additional surrounding context changes answer quality, evidence recovery, latency, and window count. Natural full-document results remain reported separately.

In [ ]:
from transformers import AutoTokenizer
from src.qasper_dataset import build_controlled_context_variants
from src.results_reporting import save_controlled_context_results

control_tokenizer = AutoTokenizer.from_pretrained(BASE_LONGFORMER_ID, use_fast=True)
controlled_sample = build_controlled_context_variants(
    sample,
    control_tokenizer,
    target_token_lengths=(384, 768, 1536, 3072, 4608),
    maximum_base_examples=12,
    seed=SEED,
)
controlled_sample.to_parquet(
    processed_dir / "qasper_controlled_context_sample.parquet", index=False
)
print("Controlled context variants:", controlled_sample.shape)
display(controlled_sample["controlled_target_tokens"].value_counts().sort_index())

controlled_scored_by_model = {}
for spec in specs:
    print(f"\n=== Controlled context: {spec.name} ===")
    runner = TransformerQABenchmarkRunner(spec)
    controlled_predictions = evaluate_runner(runner, controlled_sample)
    controlled_scored_by_model[spec.name] = score_predictions(controlled_predictions)
    runner.unload()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

controlled_comparison = save_controlled_context_results(
    controlled_scored_by_model,
    PROJECT_ROOT / "outputs",
)
display(controlled_comparison)

## 11. Review the final model comparison

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / "outputs" / "baseline_comparison.csv")
display(comparison)

display(Markdown((PROJECT_ROOT / "outputs" / "EVALUATION_REPORT.md").read_text(encoding="utf-8")))

## 12. Mandatory manual review before publishing

Open the generated per-model CSV files and inspect at least 20 incorrect or weak examples. Confirm that:

- answer text is actually supported by the predicted paragraph;
- evidence recovery is reasonable;
- failures beyond token 512 are correctly identified;
- no metrics are copied from the base model card and presented as your own;
- the README clearly distinguishes the published base checkpoint from your QASPER-fine-tuned checkpoint.

In [ ]:
output_dir = PROJECT_ROOT / "outputs"
for path in sorted(output_dir.glob("*_qa_examples.csv")):
    frame = pd.read_csv(path)
    weak = frame.sort_values(["evidence_recovered", "token_f1", "confidence_proxy"]).head(20)
    print("\n", path.name)
    display(weak[[
        "question", "reference_answers_json", "predicted_answer",
        "token_f1", "evidence_recovered", "error_category"
    ]])

## 13. Optional: upload the genuinely fine-tuned checkpoint to Hugging Face

Run this only after the training and evaluation outputs have been reviewed. Replace the repository ID with your actual Hugging Face username. This uploads the model you fine-tuned, so it can honestly be presented as a project artifact.

In [ ]:
PUSH_MODEL_TO_HUB = False
HF_MODEL_REPO_ID = "anmol-unitmole/longformer-qasper-document-qa"

if PUSH_MODEL_TO_HUB:
    from huggingface_hub import HfApi, login
    from transformers import AutoModelForQuestionAnswering, AutoTokenizer

    login()  # Enter a write token when prompted.
    model = AutoModelForQuestionAnswering.from_pretrained(FINE_TUNED_MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_MODEL_DIR)
    model.push_to_hub(HF_MODEL_REPO_ID)
    tokenizer.push_to_hub(HF_MODEL_REPO_ID)
    print("Uploaded:", HF_MODEL_REPO_ID)
else:
    print("Model upload skipped. Set PUSH_MODEL_TO_HUB=True after reviewing results.")

## 14. Files to push to GitHub

Commit the notebook, source modules, scripts, tests, generated JSON/CSV/PNG/Markdown results, updated README/model card, and the dedicated workflow. Do **not** commit the raw QASPER archives, processed full dataset, Hugging Face caches, checkpoints, or model weight files.

```bash
git add "04-long-document-question-answering-longformer" ".github/workflows/04-long-document-question-answering-longformer.yml"
git commit -m "Add QASPER fine-tuning and benchmark evaluation for Project 04"
git push origin main
```